# Architecture

This chapter is about the underlying architecture of Kafi Streams.

We first show a [dependency diagram](#dependency) of the two central classes `TopologyNode` and `Streams`.

Next, we delve deeper into the [relationship between TopologyNode and Streams](#relationship).

## Overview

* [A dependency diagram](#dependency)
  * [The TopologyNode class](#topologynode)
  * [The Streams class](#streams)
* [The relationship between TopologyNode and Streams](#relationship)


---
<a id="dependency"></a>
## A dependency diagram

Here is a high-level dependency diagram of Kafi Streams:

```mermaid
flowchart TD
    subgraph Core [ ]
        Streams["Streams class"]
        TopologyNode["TopologyNode class"]
    end

    subgraph Libs [ ]
        pydbsp["pydbsp"]
        msgpack["msgpack"]
        cloudpickle["cloudpickle"]
    end

    Streams -->|subclasses| TopologyNode

    TopologyNode ..->|imports| pydbsp
    TopologyNode ..->|imports| msgpack
    TopologyNode ..->|imports| cloudpickle

    style Core padding:30px
    style Libs padding:30px
```

The two central classes in Kafi Streams are `TopologyNode` and `Streams`.

`Streams` is a subclass of `TopologyNode`.

`TopologyNode` imports the external libraries `pydbsp`, `msgpack` and `cloudpickle`.


<a id="topologynode"></a>
### The TopologyNode class

The `TopologyNode` class wraps a fluent API on top of [*pydbsp*](https://github.com/brurucy/pydbsp) by Bruno Rucy. The fluent API is heavily inspired by the Kafka Streams DSL.

`TopologyNode` is completely abstracted away from Kafka. It does not know anything about Kafka. It just receives batches of changes, processes them relationally using pydbsp, and returns the outputs resulting from the changes. This is why it can serve also as a test harness similar to the `TopologyTestDriver` in Kafka Streams.

Stream processing in pydbsp proceeds in-memory. There are no state stores (even though pydbsp does have a pluggable storage abstraction - but Kafi Streams does not yet take advantage of it).

These are the dependencies of the TopologyNode class, from bottom to top.

#### pydbsp

The by far most important building block is pydbsp. It is the heart of Kafi Streams. It is the actual stream processing engine behind Kafi Streams.

More detailed about the interface of Kafi Streams and pydbsp can be found here: [pydbsp integration](pydbsp.ipynb).

#### msgpack

In DBSP, and hence also in pydbsp, the fundamental data type is called *Z-set*. Z-sets are modeled as Python dictionaries in pydbsp, where the keys are records/rows and the values their corresponding weights (=integers), e.g.:
```python
{"record/row_1": 1, "record/row_2": 0, "record/row_3": -1}
```

As Kafi Streams is typically used on top of Kafka, where the payloads are encoded in JSON (and Kafi converts the JSONs into Python dictionaries automatically), I needed a fast way to serialize these dictionaries into a hashable form and deserialize them back to dictionaries.

The fastest way to accomplish that is msgpack.

#### cloudpickle

cloudpickle is used for serializing/deserializing the global state of the topology (technically, the state of the corresponding pydbsp circuit) to enable [fault tolerance](checkpointing.ipynb) with checkpointing.

msgpack doesn't work since the state also contains Python objects etc., and the built-in Python pickler is also unable to serialize it - but cloudpickle is.


<a id="streams"></a>
### The Streams class

The `Streams` subclass of `TopologyNode` adds support for Kafka.

And support for [fault tolerance](checkpoints.ipynb) with checkpointing the global in-memory state to any "storage" supported by Kafi Streams, i.e., currently real Kafka or emulated Kafka on disk, S3 or Azure Blob Storage.

`Streams` implements the typical consume + process + produce loop of a stream processor:
1. It continuously consumes the source topics,
2. pushes the changes to pydbsp where it gets processed and gets the resulting changes,
3. and produces them to the sink topics.


---
<a id="relationship"></a>
## The relationship between TopologyNode and Streams

This section sheds more light on the relationship between the two main classes of Kafi Streams, `TopologyNode` and `Streams`.

The [Quickstart](quickstart.ipynb) was based on the `Streams` subclass of the `TopologyNode` class because the aim was to give you the full picture right from the start.

`Streams`, as already alluded to above, is actually just about adding support for Kafka to `TopologyNode`, and checkpointing.

The actual stream processing in Kafi Streams is completely independent of Kafka - it could, in principle, be fed by any source and emit the output to any sink.

Here is a practical example - the code from the example in [Quickstart](quickstart.ipynb), but based on `TopologyNode` instead of `Streams`:

In [ ]:
# 1. Boilerplate

!pip install -r requirements.txt

import sys
sys.path.insert(1, "../..")

from kafi.streams.topologynode import TopologyNode as Tn

from generators import ClickGenerator, CustomerGenerator
click_generator = ClickGenerator()
customer_generator = CustomerGenerator()

# 2. Specify the Topology

click_source_str = "clicks"
customer_source_str = "customers"

## a) Clicks

click_tn = (
    Tn.source(click_source_str)
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"], "ts": r["value"]["ts"]})
    .filter(lambda r: r["view_time"] > 20)
    .distinct()
)

## b) Customers

customer_tn = (
    Tn.source(customer_source_str)
    .map(lambda r: {"id": r["value"]["id"], "name": r["value"]["name"]})
    .distinct()
)

## c) Join and Sink

joined_tn = (
    click_tn
    .join(
        customer_tn,
        lambda l_r: l_r["customer_id"],
        lambda r_r: r_r["id"],
        lambda l_r, r_r: {"value": {
            "customer_id": l_r["customer_id"],
            "view_time": l_r["view_time"],
            "ts": l_r["ts"],
            "name": r_r["name"]}})
)

# 3. Build the Topology

tn = Tn.build(joined_tn)


As you can see, the topology is defined in an almost identical way. The only differences are:
* The connection to Kafka is left out, also in the source specification.
* The sources are specified using `Tn.source()` instead of `Streams.source()`.
* The sink is not explicitly specified (only possible with `TopologyNode`, not `Streams`).
* The build step is done using `Tn.build()` instead of `Streams.build()`.

Now how can supply data to the sources without Kafka? And how can we get the output of the processing?


In [ ]:
sink_m_list = []
for i in range(100):
    # 1. Generate new data.
    click_m_list = click_generator.generate(100)
    customer_m_list = customer_generator.generate(100)

    # 2. Push the new data to the topology + incrementally process the new data + get the resulting changes.
    m_list = tn.process({click_source_str: click_m_list, customer_source_str: customer_m_list})

    # 3. Add the changes to the output list.
    sink_m_list += m_list

print(len(sink_m_list))
print(sink_m_list[-10:])


In this loop, we do the following:
1. We generate new data (10.000 clicks + 10.000 customers)
2. We push the new data to the topology using the `process()` method of the `TopologyNode` class. `process()` then processes the new data and returns the resulting changes.
3. We add the changes to the output list.

That is exactly what the `Streams` subclass does, just with Kafka for sources and sinks.
